## Imports

In [62]:
import json
import time
import uuid
from datetime import datetime, timezone

import pandas as pd
from kafka import KafkaProducer

## Load Dataset

In [63]:
df = pd.read_csv("../data/raw/creditcard.csv")

print(df.shape)

df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Create Kafka Producer

In [64]:
producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

print("Kafka Producer Connected")

Kafka Producer Connected


## Stream Transactions

In [65]:
topic = "fraud-transactions"

for index, row in df.head(100).iterrows():

    transaction = row.to_dict()

    # Production metadata
    transaction["transaction_id"] = str(uuid.uuid4())

    transaction["event_timestamp"] = (
        datetime.now(timezone.utc)
        .isoformat(timespec="milliseconds")
    )

    producer.send(
        topic,
        value=transaction
    )

    print(
        f"Sent Transaction {index + 1} | "
        f"ID: {transaction['transaction_id']}"
    )

    time.sleep(1)

Sent Transaction 1 | ID: 5e033906-918b-48e2-8e33-ec8e74407ed1
Sent Transaction 2 | ID: 5196f9ec-bd3d-434a-9dee-15f726828b80
Sent Transaction 3 | ID: f383a807-3a5a-40c7-82b7-64b91c43557b
Sent Transaction 4 | ID: d4f80622-4314-4a19-a02e-1cba0ec9ead1
Sent Transaction 5 | ID: 69df7c9e-ce2f-453e-a464-91ad4c52a1c2
Sent Transaction 6 | ID: d3e6cb80-8b35-4bd0-aebd-c418e5db3720
Sent Transaction 7 | ID: c25e6850-c3cf-4e88-87bf-2bcff9cc2f3b
Sent Transaction 8 | ID: ad72e2a0-1ae1-4248-890d-9007ad7cf95f
Sent Transaction 9 | ID: 51103ee6-68b3-4bcc-990a-409f33135c30
Sent Transaction 10 | ID: 93251f5a-794a-4349-99b8-e1908c38690e
Sent Transaction 11 | ID: 8ee64e2d-37ce-4ce0-9322-5eeedfec7a83
Sent Transaction 12 | ID: 13e85013-6652-4149-91bb-40b131de065d
Sent Transaction 13 | ID: 1525491b-c44f-40fe-9520-f376b3515b4f
Sent Transaction 14 | ID: c98baebc-8896-4378-b240-6b8779adad7f
Sent Transaction 15 | ID: efa8d30f-a967-4be3-b9d8-b744c2cde6eb
Sent Transaction 16 | ID: 89fd9c37-2ed9-4de3-83a8-26139a095f0b
S

## Flush Messages

In [66]:
producer.flush()
producer.close()

print("Streaming Finished")

Streaming Finished
